# Tutorial: Custom variable extraction

This tutorial will teach you how to use the `ep.DataSet` to load the saved data from files.


In [13]:
from el_paso.typing import InternalName
from datetime import datetime, timezone

from astropy import units as u

import el_paso as ep

InternalName

ep.setup_logging()

extraction_infos = [
    ep.ExtractionInfo(
        result_key="Epoch",
        name_or_column="Epoch_Ele",
        unit=ep.units.cdf_epoch,
    ),
    ep.ExtractionInfo(
        result_key="FEDU",
        name_or_column="FEDU",
        unit=(u.cm**2 * u.s * u.sr * u.keV) ** (-1),
    ),
    ep.ExtractionInfo(
        result_key="xGEO",
        name_or_column="Position_Ele",
        unit=u.km,
    ),
]

start_time = datetime(2017, 7, 29, tzinfo=timezone.utc)
end_time = datetime(2017, 7, 30, 23, 59, 59, tzinfo=timezone.utc)

file_name_stem = "rbspa_rel04_ect-hope-pa-l3_YYYYMMDD_.{6}.cdf"

ep.download(
    start_time,
    end_time,
    save_path=".",
    download_url="https://spdf.gsfc.nasa.gov/pub/data/rbsp/rbspa/l3/ect/hope/pitchangle/rel04/YYYY/",
    file_name_stem=file_name_stem,
    file_cadence="daily",
    method="request",
    skip_existing=True,
)

variables = ep.extract_variables_from_files(
    start_time, end_time, "daily", data_path=".", file_name_stem=file_name_stem, extraction_infos=extraction_infos
)
variables

[INFO    ] 2026-05-26 13:57:36 - el_paso.download:186 - File already exists, skipping download: rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf
[INFO    ] 2026-05-26 13:57:36 - el_paso.download:186 - File already exists, skipping download: rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf
[INFO    ] 2026-05-26 13:57:36 - el_paso.download:35 - download finished in 0.503 seconds
[INFO    ] 2026-05-26 13:57:36 - el_paso.extract_variables_from_files:76 - Extracting variables ...


{'Epoch': Variable holding (8345,) data points with metadata: VariableMetadata(unit=Unit("cdf_epoch"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='', processing_notes='', standard_name=''),
 'FEDU': Variable holding (8345, 11, 72) data points with metadata: VariableMetadata(unit=Unit("1 / (keV s sr cm2)"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='', processing_notes='', standard_name=''),
 'xGEO': Variable holding (8345, 3) data points with metadata: VariableMetadata(unit=Unit("km"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='', processing_notes='', standard_name='')}

# Saving strategies

## GFZStrategy

All data at GFZ is stored under a standard, which we call `GFZStandard` and is saved using `GFZStrategy` where multiple monthly files are generated for different variables. The standard inlcudes name of the variables that are saved. All files are saved as `.mat` files. By using the `GFZStrategy`\_, the variables are automatically sorted into the corresponding files, based on the key of the variable_dict. Additionally, the variables are converted into the units as described in the standard.

## MonthlyRBStrategy

Under this strategy, the data is stored in monthly files and all the variables are written (appended) to one monthly file. In this strategy, different standards (`GFZStandard` and `PRBEMStandard`) can be used to define the name of the variables.

While using these strategies, we have to store the variables in a dictionary with certain keys, as specified by the `InternalName` type alias. These variable name and the corresponding value in the variables dictionary are statically typechecked and are also validated internally in order to refrain users from saving incorrect key and value types. We can ignore this validation by setting `ignore_validation=True` in `ep.save(..., ignore_validation=True)`


In [14]:
variables_to_save: dict[InternalName, ep.Variable] = {
    "Epoch": variables["Epoch"],
    "FEDU": variables["FEDU"],
    "Position": variables["xGEO"],
}


mrb_gfz = ep.saving_strategies.MonthlyRBStrategy(
    "./RBSP/mrb_gfz",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

gfz_gfz = ep.saving_strategies.GFZStrategy(
    "./RBSP/gfz_gfz",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)

gfz_prbem = ep.saving_strategies.GFZStrategy(
    "./RBSP/gfz_prbem",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.PRBEMStandard(),
)

mrb_prbem = ep.saving_strategies.MonthlyRBStrategy(
    "./RBSP/mrb_prbem",
    mission="RBSP",
    satellite="rbspa",
    instrument="hope",
    mag_field="T89",
    data_standard=ep.data_standards.PRBEMStandard(),
)


# ep.save(variables_to_save, mrb_gfz, start_time, end_time, time_var=variables["Epoch"])
# ep.save(variables_to_save, gfz_gfz, start_time, end_time, time_var=variables["Epoch"])
# ep.save(variables_to_save, mrb_prbem, start_time, end_time, time_var=variables["Epoch"])
# ep.save(variables_to_save, gfz_prbem, start_time, end_time, time_var=variables["Epoch"])

# Datasets

The saved datasets can be loaded using the `DataSet` class. We provide two concerete implmentation for the `DataSet`, namely `GFZDataSet` and `PRBEMDataSet`.

## GFZDataSet

`GFZDataSet/PRBEMDataSet` is used to load the data which is saved using `GFZStandard/PRBEMStandard`. The variables which can be accessed by each of these `DataSet` implementations are annotated in the respective classes in order to help user with the auto-completions.


In [15]:
from el_paso.dataset import GFZDataSet, PRBEMDataSet

mrb_gfz_data = GFZDataSet(mrb_gfz, start_time, end_time)
gfz_gfz_data = GFZDataSet(gfz_gfz, start_time, end_time)
mrb_prbem_data = PRBEMDataSet(mrb_prbem, start_time, end_time)
gfz_prbem_data = PRBEMDataSet(gfz_prbem, start_time, end_time)

[WARNING ] 2026-05-26 13:57:39 - el_paso.dataset.dataset_implementations:113 - Overriding `preferred_extension` to 'mat' since `GFZStrategy` is used, which only supports .mat files. Ignoring provided `preferred_extension` value.


In [16]:
mrb_prbem_data.metadata.datetime
mrb_prbem_data.metadata.to_dict()

[INFO    ] 2026-05-26 13:57:41 - el_paso.dataset.dataset:303 - Loading RBSP/mrb_prbem/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc


{'FEDU': VariableMetadata(unit=Unit("1 / (keV s sr cm2)"), original_cadence_seconds=np.int64(0), source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Processed unidirectional differential electron flux', processing_notes='', standard_name='FEDU'),
 'Epoch': VariableMetadata(unit=Unit("posixtime"), original_cadence_seconds=np.int64(0), source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Posix Time', processing_notes='', standard_name='Epoch'),
 'Position': VariableMetadata(unit=Unit("km"), original_cadence_seconds=np.int64(0), source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Spacecraft position in geographic cartesian coordinates', processing_notes='', standard_name='Position'),
 'datetime': VariableMetadata(unit=Unit("posixtime"), original_cadence_seconds=np.i

In [ ]:
mrb_prbem_data.Position.shape

(8345, 3)

In [ ]:
mrb_gfz_data.metadata.datetime
mrb_gfz_data.metadata.to_dict()

[INFO    ] 2026-05-26 13:49:01 - el_paso.dataset.dataset:303 - Loading RBSP/mrb_gfz/RBSP/rbspa/rbspa_hope_20170701to20170731_T89.nc


{'Flux': VariableMetadata(unit=Unit("1 / (keV s sr cm2)"), original_cadence_seconds=np.int64(0), source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Flux of particles. Can be uni/omni-directional and differential/integral.', processing_notes='', standard_name='FEDU'),
 'time': VariableMetadata(unit=Unit("datenum"), original_cadence_seconds=np.int64(0), source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Time in MATLAB datenum format.', processing_notes='', standard_name='Epoch'),
 'xGEO': VariableMetadata(unit=Unit("RE"), original_cadence_seconds=np.int64(0), source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Position in geographic cartesian coordinates.', processing_notes='', standard_name='Position'),
 'datetime': VariableMetadata(unit=Unit("datenum"), origin

In [ ]:
mrb_gfz_data.xGEO.shape

(8345, 3)

In [ ]:
gfz_gfz_data.metadata.datetime
gfz_gfz_data.metadata.to_dict()

[INFO    ] 2026-05-26 13:49:06 - el_paso.dataset.dataset:303 - Loading RBSP/gfz_gfz/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_flux_ver4.mat


{'time': VariableMetadata(unit=Unit("datenum"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Time in MATLAB datenum format.', processing_notes='', standard_name='Epoch'),
 'Flux': VariableMetadata(unit=Unit("1 / (keV s sr cm2)"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Flux of particles. Can be uni/omni-directional and differential/integral.', processing_notes='', standard_name='FEDU'),
 'datetime': VariableMetadata(unit=Unit("datenum"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Python datetime objects converted from Epoch variable. This variable is not saved to disk but computed on the fly when requested.', processing_notes='1) Computed datetime from

In [ ]:
gfz_gfz_data.xGEO.shape

[INFO    ] 2026-05-26 13:49:09 - el_paso.dataset.dataset:303 - Loading RBSP/gfz_gfz/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_xGEO_ver4.mat


(8345, 3)

In [ ]:
gfz_prbem_data.metadata.datetime
gfz_prbem_data.metadata.to_dict()

[INFO    ] 2026-05-26 13:49:13 - el_paso.dataset.dataset:303 - Loading RBSP/gfz_prbem/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_flux_ver4.mat


{'Epoch': VariableMetadata(unit=Unit("posixtime"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Posix Time', processing_notes='', standard_name='Epoch'),
 'FEDU': VariableMetadata(unit=Unit("1 / (keV s sr cm2)"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Processed unidirectional differential electron flux', processing_notes='', standard_name='FEDU'),
 'datetime': VariableMetadata(unit=Unit("posixtime"), original_cadence_seconds=0, source_files=['rbspa_rel04_ect-hope-pa-l3_20170729_v7.3.0.cdf', 'rbspa_rel04_ect-hope-pa-l3_20170730_v7.3.0.cdf'], description='Python datetime objects converted from Epoch variable. This variable is not saved to disk but computed on the fly when requested.', processing_notes='1) Computed datetime from Epoch.\n', standard_name='Epoch')}

In [ ]:
gfz_prbem_data.Position.shape

[INFO    ] 2026-05-26 13:49:23 - el_paso.dataset.dataset:303 - Loading RBSP/gfz_prbem/RBSP/rbspa/Processed_Mat_Files/rbspa_hope_20170701to20170731_xGEO_ver4.mat


(8345, 3)

# Setting data to `DataSet` externally

If in case, we want to attach some data to a `DataSet`, a complex `__setattr__` is implemented to achieve that.


In [49]:
from el_paso.dataset import DataSet

mock_strategy = ep.saving_strategies.MonthlyRBStrategy(
    base_data_path=".",
    mission="mock",
    satellite="mock",
    instrument="mock",
    mag_field="T89",
    data_standard=ep.data_standards.GFZStandard(),
)
mock_ds = DataSet(mock_strategy, start_time, end_time)

In [50]:
mock_ds.time

[WARNING ] 2026-05-26 14:06:04 - el_paso.dataset.dataset:306 - Tried to load MOCK/mock/mock_mock_20170701to20170731_T89.nc, but it does not exist


array([], dtype=float64)

In [60]:
mock_ds.time = variables["Epoch"]
mock_ds.Flux = variables["FEDU"]
mock_ds.xGEO = variables["xGEO"]

Since the `mock_ds` is initialised using `GFZStandard`, only the variables which are associated with `GFZStandard` can be attributed to it. Setting an invalid variable will raise `AttributeError`. The same applies to `PRBEMStandard`


In [ ]:
import traceback

try:
    mock_ds.Position = 1
except AttributeError as e:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_100015/1886761363.py", line 3, in <module>
    mock_ds.Position = 1
    ^^^^^^^^^^^^^^^^
  File "/home/jhawar/flag1/FLAG_PROD/code/external_data/REFACTORED_EL_PASO/el_paso/dataset/dataset.py", line 159, in __setattr__
    raise AttributeError(msg)
AttributeError: Cannot set attribute 'Position'. It is not part of GFZStandard().
